# Intent classification with DistilBERT

This hands-on section fine-tunes `distilbert-base-uncased` on Banking77: short banking questions mapped to one of 77 intents. DistilBERT is a smaller, faster student model distilled from BERT; it retains useful language representations while reducing parameters and inference cost. It is a practical speed/size trade-off, not a guarantee of equal accuracy on every task.


## Set up and check the GPU


In [4]:
# %pip install -q -r requirements.txt
# print("Dependencies installed. Restart the notebook kernel, then run the notebook from the top.")

In [5]:
import torch
from importlib.metadata import version

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Select/install a CUDA-enabled PyTorch environment "
        "using https://pytorch.org/get-started/locally/, restart the kernel, and run again."
    )

device = torch.device("cuda")
print("PyTorch:", version("torch"), "| CUDA build:", torch.version.cuda)
print("Training device:", torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PyTorch: 2.7.1+cu128 | CUDA build: 12.8
Training device: NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory: 6.0 GB


## Fine-tune and evaluate

Accuracy measures overall correctness; **macro-F1** gives each intent equal weight, making it easier to spot weak performance on less frequent classes. The test set is held out until after training.

In [6]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 42
set_seed(SEED)
dataset = load_dataset("banking77")
intent_names = dataset["train"].features["label"].names
num_labels = len(intent_names)
split = dataset["train"].train_test_split(
    test_size=0.10, seed=SEED, stratify_by_column="label"
)

model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_data = split["train"].map(tokenize_batch, batched=True, remove_columns=["text"], load_from_cache_file=False)
validation_data = split["test"].map(tokenize_batch, batched=True, remove_columns=["text"], load_from_cache_file=False)
test_data = dataset["test"].map(tokenize_batch, batched=True, remove_columns=["text"], load_from_cache_file=False)
print(f"Intents: {num_labels} | train: {len(train_data)} | validation: {len(validation_data)} | test: {len(test_data)}")

Map: 100%|██████████| 3080/3080 [00:00<00:00, 14491.75 examples/s]

Intents: 77 | train: 9002 | validation: 1001 | test: 3080


In [7]:
id2label = {i: name for i, name in enumerate(intent_names)}
label2id = {name: i for i, name in id2label.items()}
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

def compute_metrics(prediction):
    logits, labels = prediction
    predicted = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predicted),
        "macro_f1": f1_score(labels, predicted, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="intent-classifier-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=validation_data,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.395270,0.750250,0.699904
2,2.213900,0.488928,0.885115,0.871275
3,2.213900,0.376775,0.900100,0.900042
4,0.293600,0.335819,0.911089,0.911815
5,0.293600,0.337259,0.910090,0.910533


TrainOutput(global_step=1410, training_loss=0.9228868308642232, metrics={'train_runtime': 127.922, 'train_samples_per_second': 351.855, 'train_steps_per_second': 11.022, 'total_flos': 547931428084116.0, 'train_loss': 0.9228868308642232, 'epoch': 5.0})

In [8]:
test_output = trainer.predict(test_data)
true_ids = test_output.label_ids
predicted_ids = np.argmax(test_output.predictions, axis=1)
print(f"Test accuracy: {accuracy_score(true_ids, predicted_ids):.3f}")
print(f"Test macro-F1: {f1_score(true_ids, predicted_ids, average='macro'):.3f}")

matrix = confusion_matrix(true_ids, predicted_ids, labels=range(num_labels))
np.fill_diagonal(matrix, 0)
pairs = sorted(
    ((matrix[actual, predicted], actual, predicted)
     for actual in range(num_labels) for predicted in range(num_labels)),
    reverse=True,
)
print("\nMost common confusions (actual → predicted):")
for count, actual, predicted in pairs[:8]:
    if count:
        print(f"{intent_names[actual]} → {intent_names[predicted]}: {count}")

Test accuracy: 0.923
Test macro-F1: 0.923

Most common confusions (actual → predicted):
why_verify_identity → verify_my_identity: 7
get_disposable_virtual_card → getting_virtual_card: 5
transfer_into_account → topping_up_by_card: 4
card_swallowed → declined_cash_withdrawal: 4
card_arrival → card_delivery_estimate: 4
wrong_amount_of_cash_received → declined_cash_withdrawal: 3
verify_my_identity → why_verify_identity: 3
top_up_reverted → top_up_failed: 3


## Predictions and chatbot routing

The classifier's intent becomes a routing key: for example, `cash_withdrawal` can send the message to the cash-withdrawal help flow. A confidence threshold is a simple demo safeguard, not calibrated uncertainty; a real chatbot should validate it on held-out traffic and offer a clarification or human handoff.

In [9]:
def predict_intent(text, threshold=0.45):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.inference_mode():
        probabilities = torch.softmax(model(**inputs).logits, dim=-1)[0]
    confidence, label_id = probabilities.max(dim=0)
    intent = intent_names[label_id.item()]
    if confidence.item() < threshold:
        return {"message": text, "route": "ask_clarifying_question", "prediction": intent,
                "confidence": round(confidence.item(), 3)}
    return {"message": text, "route": intent, "confidence": round(confidence.item(), 3)}

examples = [
    "I lost my card and need to freeze it",
    "How long will my bank transfer take?",
    "I forgot my PIN",
    "There is a charge on my statement I don't recognize",
    "I want to close my account",
]
for message in examples:
    print(predict_intent(message))

{'message': 'I lost my card and need to freeze it', 'route': 'lost_or_stolen_card', 'confidence': 0.913}
{'message': 'How long will my bank transfer take?', 'route': 'balance_not_updated_after_bank_transfer', 'confidence': 0.94}
{'message': 'I forgot my PIN', 'route': 'pin_blocked', 'confidence': 0.927}
{'message': "There is a charge on my statement I don't recognize", 'route': 'card_payment_not_recognised', 'confidence': 0.956}
{'message': 'I want to close my account', 'route': 'terminate_account', 'confidence': 0.982}


## LLM-powered chatbot responses

The intent classifier tells us *what* the customer wants; now an LLM generates a helpful *reply*. The pipeline is:

1. **Classify** the message with DistilBERT → get an intent and confidence score.
2. **Build a system prompt** that tells the LLM its role and the detected intent so the response stays on-topic.
3. **Call the LLM** (Nvidia Nemotron 3.5 Lightning via OpenRouter) with reasoning enabled to produce a natural-language answer.

This keeps the classifier's speed and measurability (intent labels, confidence, confusion matrix) while adding the fluency of a large language model. Put your OpenRouter API key in `.streamlit/secrets.toml` before running.

In [10]:
import requests as req
import json
try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib  # Python < 3.11 fallback

with open(".streamlit/secrets.toml", "rb") as f:
    _secrets = tomllib.load(f)
OPENROUTER_API_KEY = _secrets["OPENROUTER_API_KEY"]
LLM_MODEL = "nvidia/nemotron-3.5-lightning:free"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

def call_llm(messages, reasoning=True):
    """Send messages to the OpenRouter API and return the assistant reply."""
    payload = {
        "model": LLM_MODEL,
        "messages": messages,
    }
    if reasoning:
        payload["reasoning"] = {"enabled": True}

    response = req.post(
        url=OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        data=json.dumps(payload),
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]


def chatbot_respond(user_message):

    # Step 1 – classify intent
    classification = predict_intent(user_message)
    intent = classification["route"]
    confidence = classification["confidence"]

    # Step 2 – build system prompt scoped to the detected intent
    system_prompt = (
        f"You are a helpful banking support assistant. "
        f"The customer's message has been classified as intent: '{intent}' "
        f"(confidence: {confidence:.0%}). "
        f"Respond concisely and helpfully to their query. "
        f"If the intent is 'ask_clarifying_question' the classifier was "
        f"unsure, so politely ask the customer to rephrase or clarify."
    )

    # Step 3 – call the LLM
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    assistant_msg = call_llm(messages)

    return {
        "intent": intent,
        "confidence": confidence,
        "reply": assistant_msg.get("content", ""),
    }

print("chatbot_respond() ready.")

chatbot_respond() ready.


In [11]:
# Demo: classify + generate for each example message
for message in examples:
    result = chatbot_respond(message)
    print(f"Customer : {message}")
    print(f"Intent   : {result['intent']}  (confidence {result['confidence']:.0%})")
    print(f"Bot      : {result['reply']}")
    print("-" * 72)

Customer : I lost my card and need to freeze it
Intent   : lost_or_stolen_card  (confidence 91%)
Bot      : I'm sorry your card is lost. You can freeze it instantly through our mobile app or online banking under "Card Controls." If you need help accessing that feature or prefer to speak with someone, just let me know and I'll provide your options. 

Once frozen, we'll automatically send a replacement card to your address on file. Also, keep an eye on your account for any unauthorized activity and contact us right away if you spot anything suspicious.
------------------------------------------------------------------------
Customer : How long will my bank transfer take?
Intent   : balance_not_updated_after_bank_transfer  (confidence 94%)
Bot      : Bank transfers typically take 1–3 business days to process, depending on the banks involved, transfer type, and whether it's domestic or international. If your balance hasn't updated yet, it may still be processing, or the delay could be due 

## Wrap-up
The recommendation case study shows why a measurable engagement reward can shape what people see, and why diversity, safety, and user value matter alongside it. The classifier demo shows the other side of applied ML: a specific, bounded prediction (intent) can route a request, with evaluation and a fallback for uncertain predictions. The LLM chatbot section demonstrates how the classifier's structured output (intent label, confidence) can drive a large language model to produce fluent, on-topic replies. Record the actual GPU name and test metrics from your run in your presentation.